# Radio Modulation Classification with Deep Learning

**Can a CNN learn to identify how a radio signal was modulated — directly from raw IQ samples, with no hand-crafted features?**

This notebook walks through the full pipeline:
1. Understanding the data and the SNR challenge
2. Visualizing modulation signatures in the IQ plane
3. Building and training a 1D CNN
4. Evaluating performance as a function of SNR

Dataset: [RadioML 2016.10a](https://www.deepsig.ai/datasets) — 220,000 IQ samples across 11 modulation types and 20 SNR levels.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

from src.dataset import RadioMLDataset, get_dataloaders, MODULATIONS, MOD_TO_IDX, SNR_VALUES
from src.model import ModulationCNN
from src.train import train, evaluate
from src.evaluate import accuracy_vs_snr, plot_accuracy_vs_snr, plot_confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')
FIGDIR = '../results/figures'
DATA_PATH = '../data/RML2016.10a_dict.pkl'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

## 2. Data Exploration

### 2.1 Dataset Overview

RadioML 2016.10a is a synthetic dataset generated with GNU Radio, simulating realistic channel impairments: noise, fading, carrier frequency offset, and sample rate offset. Each sample is a sequence of 128 complex IQ (in-phase/quadrature) observations, stored as a `(2, 128)` float array.

The dataset spans:
- **11 modulation types**: analog (AM-DSB, AM-SSB, WBFM) and digital (BPSK, QPSK, 8PSK, QAM16, QAM64, PAM4, CPFSK, GFSK)
- **20 SNR levels**: −20 dB to +18 dB in steps of 2 dB
- **1,000 samples** per (modulation, SNR) pair → 220,000 total

In [ ]:
with open(DATA_PATH, 'rb') as f:
    raw = pickle.load(f, encoding='latin1')

print(f'Keys (mod, snr): {len(raw)}')
print(f'Sample shape   : {raw[("BPSK", 0)].shape}')
print(f'Modulations    : {MODULATIONS}')
print(f'SNR range      : {SNR_VALUES[0]} dB to {SNR_VALUES[-1]} dB')

### 2.2 Class Distribution

The dataset is perfectly balanced — 20,000 samples per modulation class.

In [ ]:
counts = {mod: sum(1000 for snr in SNR_VALUES) for mod in MODULATIONS}

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(MODULATIONS, [counts[m] for m in MODULATIONS], color='steelblue', edgecolor='white')
ax.set_ylabel('Number of samples', fontsize=12)
ax.set_title('Samples per Modulation Class', fontsize=13)
ax.set_ylim(0, 25000)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{int(bar.get_height()/1000)}K', ha='center', va='bottom', fontsize=9)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{FIGDIR}/class_distribution.png', dpi=150)
plt.show()

### 2.3 IQ Time Series — What Does Each Modulation Look Like?

Each sample is a pair of time series: the **I** (in-phase) and **Q** (quadrature) components of the received signal. At high SNR, the modulation structure is visible. At low SNR, noise dominates.

In [ ]:
SNR_SHOW = 18  # high SNR so structure is visible

def normalize_iq(sample):
    """Normalize IQ sample to unit RMS power for visualization."""
    rms = np.sqrt(np.mean(sample ** 2))
    return sample / (rms + 1e-8)

fig, axes = plt.subplots(11, 2, figsize=(12, 22))
fig.suptitle(f'IQ Time Series per Modulation (SNR = {SNR_SHOW} dB)', fontsize=14, y=1.01)

for row, mod in enumerate(MODULATIONS):
    sample = normalize_iq(raw[(mod, SNR_SHOW)][0])  # shape (2, 128), normalized
    axes[row, 0].plot(sample[0], linewidth=0.9, color='steelblue')
    axes[row, 1].plot(sample[1], linewidth=0.9, color='coral')
    axes[row, 0].set_ylabel(mod, fontsize=9, rotation=0, labelpad=45)
    for ax in axes[row]:
        ax.set_ylim(-3, 3)
        ax.tick_params(labelsize=7)

axes[0, 0].set_title('I channel', fontsize=11)
axes[0, 1].set_title('Q channel', fontsize=11)
axes[-1, 0].set_xlabel('Time sample', fontsize=10)
axes[-1, 1].set_xlabel('Time sample', fontsize=10)

plt.tight_layout()
plt.savefig(f'{FIGDIR}/iq_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 The SNR Challenge

The central difficulty of this problem: as SNR drops, the signal structure disappears into noise. Below we show the same QPSK signal across the full SNR range.

In [ ]:
MOD_SHOW = 'QPSK'
snrs_to_show = [-20, -10, 0, 10, 18]

fig, axes = plt.subplots(len(snrs_to_show), 1, figsize=(12, 8), sharex=True)
fig.suptitle(f'{MOD_SHOW} — I Channel Across SNR Levels', fontsize=13)

for ax, snr in zip(axes, snrs_to_show):
    sample = normalize_iq(raw[(MOD_SHOW, snr)][0])
    ax.plot(sample[0], linewidth=0.9, color='steelblue')
    ax.set_ylabel(f'{snr} dB', fontsize=9, rotation=0, labelpad=35)
    ax.set_ylim(-3, 3)
    ax.tick_params(labelsize=8)

axes[-1].set_xlabel('Time sample', fontsize=10)
plt.tight_layout()
plt.savefig(f'{FIGDIR}/snr_effect.png', dpi=150)
plt.show()

### 2.5 Constellation Diagrams

Plotting I vs Q for each modulation reveals the geometric structure that encodes information. Digital modulations have discrete constellation points; analog modulations spread continuously. At high SNR these patterns are clean — and this is exactly what the CNN will learn to recognize.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle(f'Constellation Diagrams (SNR = {SNR_SHOW} dB)', fontsize=14)
axes_flat = axes.flatten()

for idx, mod in enumerate(MODULATIONS):
    ax = axes_flat[idx]
    samples = raw[(mod, SNR_SHOW)]          # (1000, 2, 128)
    # normalize each sample to unit RMS power before plotting
    rms = np.sqrt(np.mean(samples ** 2, axis=(1, 2), keepdims=True))
    samples_norm = samples / (rms + 1e-8)
    i_vals = samples_norm[:, 0, :].flatten()
    q_vals = samples_norm[:, 1, :].flatten()
    ax.scatter(i_vals, q_vals, s=0.3, alpha=0.2, color='steelblue', rasterized=True)
    ax.set_title(mod, fontsize=10)
    lim = np.percentile(np.abs(np.concatenate([i_vals, q_vals])), 99) * 1.2
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.tick_params(labelsize=7)

axes_flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig(f'{FIGDIR}/constellations.png', dpi=150)
plt.show()

## 3. Model

### Architecture

The input to the model is a raw `(2, 128)` IQ tensor — no hand-crafted features, no preprocessing beyond loading. We treat the I and Q channels as two input channels and apply 1D convolutions along the time axis.

```
Input  (2 × 128)
  └── Block 1: Conv1D(64) → BN → ReLU → Conv1D(64) → BN → ReLU → MaxPool → Dropout
  └── Block 2: Conv1D(128) → BN → ReLU → Conv1D(128) → BN → ReLU → MaxPool → Dropout
  └── Block 3: Conv1D(256) → BN → ReLU → GlobalAvgPool
  └── FC(256→128) → ReLU → Dropout → FC(128→11)
```

Global average pooling at the end makes the model agnostic to the exact sequence length and reduces the number of parameters compared to a fully-connected flattening. BatchNorm after each convolution stabilizes training; Dropout prevents overfitting at low-SNR where the model could memorize noise patterns.

In [ ]:
model = ModulationCNN(num_classes=len(MODULATIONS))

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

In [ ]:
# Verify input/output shapes
dummy = torch.randn(8, 2, 128)
out = model(dummy)
print(f'Input shape : {dummy.shape}')
print(f'Output shape: {out.shape}  (batch × num_classes)')

## 4. Training

We split the dataset 80/10/10 into train/val/test. The model is trained with:
- **Optimizer**: Adam (lr = 1e-3)
- **LR schedule**: Cosine annealing over 30 epochs
- **Loss**: Cross-entropy
- **Batch size**: 256

The best checkpoint (highest val accuracy) is saved and used for all evaluation below.

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(DATA_PATH, batch_size=256)
print(f'Train: {len(train_loader.dataset):,} samples')
print(f'Val  : {len(val_loader.dataset):,} samples')
print(f'Test : {len(test_loader.dataset):,} samples')

In [ ]:
history = train(
    model, train_loader, val_loader,
    epochs=30,
    lr=1e-3,
    device=DEVICE,
    checkpoint_dir='../checkpoints'
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs, history['val_loss'], label='Val', color='coral')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()

axes[1].plot(epochs, [a * 100 for a in history['val_acc']], color='steelblue')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Validation Accuracy')

plt.tight_layout()
plt.savefig(f'{FIGDIR}/training_curves.png', dpi=150)
plt.show()

## 5. Evaluation

### 5.1 Overall Test Accuracy

We load the best checkpoint and evaluate on the held-out test set.

In [ ]:
import torch.nn as nn

model.load_state_dict(torch.load('../checkpoints/best_model.pt', map_location=DEVICE))
model.to(DEVICE)

_, test_acc = evaluate(model, test_loader, nn.CrossEntropyLoss(), DEVICE)
print(f'Overall test accuracy: {test_acc*100:.1f}%')

### 5.2 Accuracy vs. SNR

Overall accuracy is misleading here — a model that performs well at +18 dB but fails at −10 dB is only half-useful. The key plot is accuracy as a function of SNR.

We expect near-random performance at very low SNR (−20 dB), with accuracy rising steeply above 0 dB as signal structure becomes distinguishable from noise.

In [ ]:
results = accuracy_vs_snr(model, DATA_PATH, device=DEVICE)
fig = plot_accuracy_vs_snr(results, save_path=f'{FIGDIR}/accuracy_vs_snr.png')
plt.show()

print('\nPer-SNR breakdown:')
for snr in sorted(results):
    bar = '█' * int(results[snr] * 40)
    print(f'  {snr:+4d} dB  {results[snr]*100:5.1f}%  {bar}')

### 5.3 Confusion Matrix (High-SNR Regime)

At high SNR the model should have high confidence. The confusion matrix reveals which modulations are systematically confused — typically those with similar constellation geometry (e.g. BPSK vs. QPSK at intermediate SNR, or analog modulations).

In [ ]:
fig = plot_confusion_matrix(
    model, DATA_PATH,
    snr_min=0,
    device=DEVICE,
    save_path=f'{FIGDIR}/confusion_matrix.png'
)
plt.show()

## 7. Discussion

### Results Summary

| Model | Avg (all SNR) | Avg (SNR ≥ 0 dB) |
|---|---|---|
| Baseline | 60.3% | **88.0%** |
| A: Augmentation | 58.6% | 86.9% |
| B: Expert features | 57.8% | 84.2% |
| C: Focal loss | 57.8% | 83.0% |
| D: All combined | 57.3% | 82.8% |

The baseline outperforms all variants at 30 epochs. This is a meaningful finding in itself.

### Why the improvements didn't help

**Phase rotation augmentation** introduces harder training examples — the model sees more variation per epoch, which requires more epochs to converge. At 30 epochs the regularization cost outweighs the generalization benefit. With 60–100 epochs it would likely pull ahead.

**Expert features** (instantaneous amplitude and phase) add no new information the CNN can't already extract. A convolutional network applied to raw IQ implicitly learns to compute amplitude and phase-like statistics in its first layers. Feeding them explicitly doubles the input redundancy and adds noise to the optimization without giving the model anything it didn't already have.

**Focal loss** with gamma=2.0 is too aggressive for this dataset. It over-penalizes confident predictions on easy classes (CPFSK, GFSK, BPSK) to focus on the hard ones (QAM16/QAM64), but the easy classes make up the bulk of correct predictions — hurting them tanks the overall number without meaningfully fixing the QAM confusion.

### The QAM16/QAM64 ceiling

The 88% plateau is not a model limitation — it's a data limitation. QAM16 and QAM64 share the same square grid geometry; the only difference is constellation density. At SNR ≥ 0 dB, residual noise still blurs the fine point spacing that separates them, and 128 IQ samples is a short observation window to resolve that difference reliably. No technique tested here addresses that root cause.

### What would actually help

**Quick experiments worth trying:**
- **Tune focal loss gamma** — lower values (gamma=0.5 or 1.0) are less aggressive and may improve over baseline without over-penalizing easy classes
- **More epochs with augmentation** — running Variant A for 60–100 epochs would likely recover and surpass the baseline, since the regularization benefit of phase rotation needs time to compound
- **Learning rate sweep** — a slightly lower initial LR (5e-4) combined with augmentation often stabilizes convergence on harder training distributions

**Larger architectural changes:**
- **Longer input sequences** — 512 or 1024 IQ samples instead of 128 would give the model more evidence per classification decision, directly helping QAM disambiguation
- **Higher-order cumulants** — 4th and 6th order cumulants are provably discriminative for QAM orders and have been the backbone of classical AMC for decades; a hybrid model combining cumulant features with a CNN backbone is a natural next step
- **Recurrent or Transformer backbone** — an LSTM or attention mechanism over the IQ sequence can capture longer-range symbol patterns that a purely local CNN misses, particularly for analog modulations like WBFM
- **SNR estimation as an auxiliary task** — training the model to jointly predict modulation type and SNR level encourages it to learn SNR-aware representations, which improves low-SNR accuracy

## 6. Improving the Baseline

The baseline model plateaus at ~86% above 0 dB with two clear failure modes: QAM16/QAM64 confusion and WBFM/AM-DSB confusion. We test three targeted improvements:

| Variant | Idea |
|---|---|
| **A — Augmentation** | Random phase rotation during training forces rotation-invariant features |
| **B — Expert features** | Instantaneous amplitude & phase added as input channels — classical AMC features |
| **C — Focal loss** | Down-weights easy classes, focuses training on QAM16/QAM64 confusion |
| **D — All combined** | All three together |

All variants use the same architecture, split, seed, and 30 epochs for fair comparison.

### 6.1 Variant A — Phase Rotation Augmentation

In a real receiver, the carrier phase offset is unknown — the signal may arrive rotated by any arbitrary angle in the IQ plane. By randomly rotating each training sample, we force the model to learn features that are invariant to phase offset, rather than memorizing the absolute orientation of constellation points.

In [ ]:
from src.model import ModulationCNN
from src.train import train

train_A, val_A, test_A = get_dataloaders(DATA_PATH, batch_size=256, augment=True)
model_A = ModulationCNN(num_classes=len(MODULATIONS), in_channels=2).to(DEVICE)

history_A = train(
    model_A, train_A, val_A,
    epochs=30, lr=1e-3, device=DEVICE,
    checkpoint_dir='../checkpoints',
    checkpoint_name='model_A_augment.pt'
)

### 6.2 Variant B — Expert Features

Instead of raw IQ alone, we append two classical AMC features computed from the IQ signal:
- **Instantaneous amplitude**: `√(I² + Q²)` — encodes the signal envelope
- **Instantaneous phase**: `arctan(Q/I)` — encodes phase transitions between symbols

These features are especially discriminative for QAM orders — QAM64 has more amplitude levels than QAM16, and the phase transitions have different statistics. The model input becomes `(4, 128)` instead of `(2, 128)`.

In [ ]:
train_B, val_B, test_B = get_dataloaders(DATA_PATH, batch_size=256, expert_features=True)
model_B = ModulationCNN(num_classes=len(MODULATIONS), in_channels=4).to(DEVICE)

history_B = train(
    model_B, train_B, val_B,
    epochs=30, lr=1e-3, device=DEVICE,
    checkpoint_dir='../checkpoints',
    checkpoint_name='model_B_expert.pt'
)

### 6.3 Variant C — Focal Loss

Standard cross-entropy treats all misclassifications equally. Focal loss adds a modulating factor `(1-p)^γ` that down-weights confident correct predictions and focuses the gradient on hard, uncertain examples — in our case, QAM16/QAM64 pairs that the model consistently confuses.

In [ ]:
from src.focal_loss import FocalLoss

train_C, val_C, test_C = get_dataloaders(DATA_PATH, batch_size=256)
model_C = ModulationCNN(num_classes=len(MODULATIONS), in_channels=2).to(DEVICE)

history_C = train(
    model_C, train_C, val_C,
    epochs=30, lr=1e-3, device=DEVICE,
    checkpoint_dir='../checkpoints',
    checkpoint_name='model_C_focal.pt',
    criterion=FocalLoss(gamma=2.0)
)

### 6.4 Variant D — All Combined

In [ ]:
train_D, val_D, test_D = get_dataloaders(DATA_PATH, batch_size=256, augment=True, expert_features=True)
model_D = ModulationCNN(num_classes=len(MODULATIONS), in_channels=4).to(DEVICE)

history_D = train(
    model_D, train_D, val_D,
    epochs=30, lr=1e-3, device=DEVICE,
    checkpoint_dir='../checkpoints',
    checkpoint_name='model_D_combined.pt',
    criterion=FocalLoss(gamma=2.0)
)

### 6.5 Comparison — Accuracy vs. SNR

All five models evaluated on the same held-out test set.

In [ ]:
import torch.nn as nn

# load best checkpoints
model_A.load_state_dict(torch.load('../checkpoints/model_A_augment.pt',  map_location=DEVICE))
model_B.load_state_dict(torch.load('../checkpoints/model_B_expert.pt',   map_location=DEVICE))
model_C.load_state_dict(torch.load('../checkpoints/model_C_focal.pt',    map_location=DEVICE))
model_D.load_state_dict(torch.load('../checkpoints/model_D_combined.pt', map_location=DEVICE))

# evaluate each model — note: B and D use expert_features=True
results_A = accuracy_vs_snr(model_A, DATA_PATH, device=DEVICE)
results_B = accuracy_vs_snr(model_B, DATA_PATH, device=DEVICE, expert_features=True)
results_C = accuracy_vs_snr(model_C, DATA_PATH, device=DEVICE)
results_D = accuracy_vs_snr(model_D, DATA_PATH, device=DEVICE, expert_features=True)

# comparison plot
snrs = sorted(results.keys())
variants = {
    'Baseline':              (results,   'steelblue',  '-'),
    'A: Augmentation':       (results_A, 'coral',      '--'),
    'B: Expert features':    (results_B, 'seagreen',   '--'),
    'C: Focal loss':         (results_C, 'mediumpurple','--'),
    'D: All combined':       (results_D, 'crimson',    '-'),
}

fig, ax = plt.subplots(figsize=(11, 5))
for label, (res, color, ls) in variants.items():
    accs = [res[s] * 100 for s in snrs]
    ax.plot(snrs, accs, label=label, color=color, linestyle=ls,
            linewidth=2, marker='o', markersize=4)

ax.axhline(100 / 11, color='gray', linestyle=':', linewidth=1, label='Random chance (9.1%)')
ax.set_xlabel('SNR (dB)', fontsize=13)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('Accuracy vs. SNR — Baseline vs. Improvements', fontsize=14)
ax.set_xticks(snrs)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGDIR}/accuracy_vs_snr_comparison.png', dpi=150)
plt.show()

# summary table
print(f"\n{'Model':<25} {'Avg (all SNR)':>14} {'Avg (SNR≥0)':>12}")
print('-' * 53)
for label, (res, _, _) in variants.items():
    avg_all  = np.mean([res[s] for s in snrs]) * 100
    avg_high = np.mean([res[s] for s in snrs if s >= 0]) * 100
    print(f'{label:<25} {avg_all:>13.1f}%  {avg_high:>10.1f}%')